# Phase 3: Advanced RAG Engine with Conversational Memory (V1)

In this final phase, I am implementing a **History-Aware RAG Engine**. The goal is to allow the AI to maintain context across a conversation, enabling it to answer follow-up questions and understand references like "it" or "that" based on previous chat turns.

### Key Features:
* **Persistent Chat History:** Storing previous interactions using LangChain's message structures.
* **Contextual Re-questioning:** Re-writing user queries based on history for better document retrieval.
* **Seamless PDF Integration:** Full RAG pipeline with Llama 3.3 and FAISS.

## 1. Environment Installation
Ensuring all core libraries are updated and compatible.

In [2]:
!pip install -qU \
    langchain \
    langchain-groq \
    langchain-community \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu \
    pypdf

## 2. Infrastructure Setup
Importing modules for chat memory, document processing, and AI orchestration.

In [1]:
import os
from getpass import getpass
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# API Authentication
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

## 3. Knowledge Base Preparation
Loading the corporate document and converting it into a searchable vector format.

In [10]:
# Processing the document
loader = PyPDFLoader("SOP_Company.pdf")
pages = loader.load()

# Splitting into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = splitter.split_documents(pages)

# Initializing Vector Store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

print(f"Knowledge Base ready with {len(docs)} document chunks.")

## 4. Designing the Conversational Chain
I am designing a prompt that can handle both the retrieved context and the conversation history.

In [8]:
# Defining the System Prompt with Memory
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a professional corporate assistant. Use the provided context and conversation history to answer the user's question accurately."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# The Advanced RAG Pipeline
# Note: In a real-world app, 'input' will be used to search the 'retriever'
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough(), "chat_history": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

## 5. Simulating a Multi-turn Conversation
Testing the AI's ability to remember previous statements.

In [ ]:
chat_history = []

# Turn 1
user_input_1 = "What is the policy for being late?"
response_1 = rag_chain.invoke({
    "input": user_input_1,
    "chat_history": chat_history,
    "context": retriever.invoke(user_input_1)
})

print(f"Rado: {user_input_1}")
print(f"AI: {response_1}\n")

# Adding to History
chat_history.extend([
    HumanMessage(content=user_input_1),
    AIMessage(content=response_1)
])

# Turn 2 (Follow-up)
user_input_2 = "What if it happens twice in a week?"
response_2 = rag_chain.invoke({
    "input": user_input_2,
    "chat_history": chat_history,
    "context": retriever.invoke(user_input_2)
})

print(f"Rado: {user_input_2} (Follow-up)")
print(f"AI: {response_2}")